#SLEAP_AI.ipnyb

#Description
---------
This notebook runs SLEAP.ai in the colab Virtual Machine.

It uses the power of NVIDIA Tesla T4 GPU to take the best out of SLEAP running it's deep learning neural networks faster.

The system used is condacolab since it helps bypass default python limitations from colab

SLEAP is run inside an environment where it has the necessary drivers.

Run order

1.   Choose the Paths.
2.   Prepare the environment
3.   Run SLEAP
4.   Output the predictions



SLEAP.ai DOI: https://doi.org/10.1038/s41592-022-01426-1

Moita Lab · Champalimaud Foundation: https://moitalab.org/

Rodrigo Garrido

#Define Paths

In [ ]:
import os
from IPython.display import display, clear_output
import ipywidgets as widgets
from google.colab import drive
from ipyfilechooser import FileChooser

# 1. Mount Google Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. Repository Settings (Fixed Model)
REPO_URL = "https://github.com/moitalab/Sleap_Colab.git"
REPO_DIR = "/content/Sleap_Colab"

# Automatic repository download (including best_model.h5 via LFS)
if not os.path.exists(REPO_DIR):
    print(" Downloading model and scripts from GitHub (Moita Lab)...")
    !git clone {REPO_URL}
else:
    print(" GitHub model is already prepared.")

# --- SELECTION INTERFACE ---
chooser_exp = FileChooser('/content/drive/MyDrive/')
chooser_exp.title = '<b>Select Experiment Folder (Root)</b>'
chooser_exp.show_only_dirs = True

save_button = widgets.Button(
    description='Confirm Configuration',
    button_style='success',
    icon='check',
    layout=widgets.Layout(width='300px')
)

output_log = widgets.Output()

def on_confirm_clicked(b):
    with output_log:
        clear_output()
        exp_root = chooser_exp.selected_path
        if not exp_root:
            print(" Error: Please select the experiment folder in Drive!")
            return

        # Standard repository subfolder paths
        crop_raw = os.path.join(exp_root, "PostProcessing", "CropRaw")
        arenas = os.path.join(exp_root, "PostProcessing", "Arenas")
        pose_folder = os.path.join(exp_root, "PostProcessing", "Pose")
        tracked_folder = os.path.join(exp_root, "PostProcessing", "Tracked")


        # Ensure the output directory exists
        os.makedirs(pose_folder, exist_ok=True)

        # Save paths for the Bash environment
        with open('/content/sleap_paths.env', 'w') as f:
            f.write(f'export VIDEO_FOLDER="{crop_raw}"\n')
            f.write(f'export ARENAS_FOLDER="{arenas}"\n')
            f.write(f'export OUTPUT_FOLDER="{pose_folder}"\n')
            f.write(f'export TRACKED_FOLDER="{tracked_folder}"\n')
            f.write(f'export MODEL_PATH="{REPO_DIR}"\n')

        print("-" * 50)
        print(" PIPELINE CONFIGURED!")
        print(f" Model: Loaded from GitHub")
        print(f" Tracked Data (Bonsai): {tracked_folder}") # Log for verification
        print(f" Videos: {crop_raw}")
        print("-" * 50)

save_button.on_click(on_confirm_clicked)
display(chooser_exp, save_button, output_log)

#Enviroment preparation

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install() # Runtime will restart

In [ ]:
!mamba create -y -n sleap_env -c conda-forge -c nvidia -c sleap -c anaconda \
    python=3.7 \
    sleap=1.3.4 \
    tensorflow=2.7.0 \
    cudatoolkit=11.3.1 \
    cudnn=8.2.1.32 \
    pandas=1.3.5 \
    scipy=1.7.3 \
    matplotlib=3.5.3

In [ ]:
%%bash

# Test if th gpu is available
source activate sleap_env
python -c "import sleap; print('SLEAP Version Check:', sleap.__version__)"
python -c "import tensorflow as tf; print('Num GPUs Available: ', len(tf.config.list_physical_devices('GPU')))"

#Running


In [ ]:
%%bash
# =========================================================
# SLEAP Inference + Post-processing  (8-Keypoint model)
# =========================================================

# --- Verify that Cell 1 was executed after kernel restart ---
if [ ! -f /content/sleap_paths.env ]; then
    echo ''
    echo 'ERROR: /content/sleap_paths.env not found!'
    echo 'After kernel restart (condacolab), rerun'
    echo 'Cell 1 (Define Paths) before running this cell.'
    exit 1
fi

source /content/sleap_paths.env
source activate sleap_env

echo '--- Configured Paths ---'
echo "  VIDEO_FOLDER   : $VIDEO_FOLDER"
echo "  ARENAS_FOLDER  : $ARENAS_FOLDER"
echo "  TRACKED_FOLDER : $TRACKED_FOLDER"
echo "  MODEL_PATH     : $MODEL_PATH"
echo ''

if [ -z "$VIDEO_FOLDER" ]; then
    echo 'ERROR: VIDEO_FOLDER not defined. Rerun Cell 1.'
    exit 1
fi

if [ ! -d "$VIDEO_FOLDER" ]; then
    echo "ERROR: Video folder not found: $VIDEO_FOLDER"
    exit 1
fi

N_VIDEOS=$(ls "$VIDEO_FOLDER"/*.avi 2>/dev/null | wc -l)
echo ".avi videos found: $N_VIDEOS"
if [ "$N_VIDEOS" -eq 0 ]; then
    echo "ERROR: No .avi files in $VIDEO_FOLDER"
    echo 'Check if the CropRaw folder contains the videos.'
    exit 1
fi

POSE_TEMP="/content/pose_temp"
mkdir -p "$POSE_TEMP"

# --- 1. SLEAP Tracking ---
echo '=== SLEAP Inference ==='
for video in "$VIDEO_FOLDER"/*.avi; do
    [ -f "$video" ] || continue
    c_id=$(basename "$video" .avi)
    slp_out="$POSE_TEMP/${c_id}.predictions.slp"

    if [ -f "$slp_out" ]; then
        echo "  [skip]  $c_id  (already processed)"
    else
        echo "  [track] $c_id ..."
        sleap-track "$video" --model "$MODEL_PATH" -o "$slp_out" --no-empty-frames --verbosity json
        if [ $? -ne 0 ]; then
            echo "  [ERROR] sleap-track failed for $c_id"
        fi
    fi
done

echo ''
echo '=== Post-processing: SLEAP + Bonsai ==='

python3 << 'PYEOF'
import os, glob
import sleap
import pandas as pd
import numpy as np
from PIL import Image

arenas_dir     = os.environ['ARENAS_FOLDER']
output_dir     = os.environ['OUTPUT_FOLDER']
tracked_folder = os.environ['TRACKED_FOLDER']
pose_temp      = "/content/pose_temp"

node_map = {
    'L': 'Left', 'R': 'Right', 'H': 'Head', 'Trx': 'Thorax',
    'Abd': 'Abdomen', 'Lw': 'LeftWing', 'Rw': 'RightWing', 'T': 'Top'
}

slp_files = sorted(glob.glob(os.path.join(pose_temp, "*.predictions.slp")))

if not slp_files:
    print("ERROR: No .slp files found in", pose_temp)
    print("sleap-track produced no output. Check error messages above.")
    exit(1)

for slp_path in slp_files:
    c_id = os.path.basename(slp_path).replace(".predictions.slp", "")

    # Bonsai CSV name: replace _crop with _tracked
    tracked_id = c_id.replace("_crop", "_tracked")
    csv_path = os.path.join(tracked_folder, f"{tracked_id}.csv")
    if not os.path.exists(csv_path):
        print(f"  [skip] Missing Bonsai CSV for {c_id}  (expected: {tracked_id}.csv)")
        continue

    print(f"  Processing: {c_id}")

    session_prefix = c_id.split('-fly')[0].split('_fly')[0]
    arena_candidates = [
        os.path.join(arenas_dir, f"{c_id.replace('_crop', '')}.png"),
        os.path.join(arenas_dir, f"{session_prefix}.png"),
    ]
    arena_img = next((p for p in arena_candidates if os.path.exists(p)), None)

    if arena_img:
        with Image.open(arena_img) as img:
            w_arena, h_arena = float(img.size[0]), float(img.size[1])
    else:
        tried = [os.path.basename(p) for p in arena_candidates]
        print(f"    Arena image not found (tried: {tried}) -- using 1280x1024.")
        w_arena, h_arena = 1280.0, 1024.0

    df_t   = pd.read_csv(csv_path)
    labels = sleap.load_file(slp_path)

    col_x = [c for c in df_t.columns if 'X' in c.upper() and ('CENTROID' in c.upper() or len(c) == 1)][0]
    col_y = [c for c in df_t.columns if 'Y' in c.upper() and ('CENTROID' in c.upper() or len(c) == 1)][0]

    data = []
    for frame in labels:
        f_idx = frame.frame_idx
        c_row = df_t[df_t['FrameIndex'] == f_idx]
        if c_row.empty:
            continue

        cx_px = c_row[col_x].values[0] * w_arena
        cy_px = c_row[col_y].values[0] * h_arena
        row   = {'FrameIndex': f_idx}

        if len(frame.instances) == 0:
            for name in node_map.values():
                row[f'{name}.Position.X'] = float('nan')
                row[f'{name}.Position.Y'] = float('nan')
                row[f'{name}.Confidence'] = 0.0
        else:
            for inst in frame.instances:
                for node in labels.skeleton.nodes:
                    pt   = inst[node.name]
                    name = node_map.get(node.name, node.name)
                    if pt is not None:
                        norm_x = (cx_px + (pt.x - 64.0)) / w_arena
                        norm_y = (cy_px + (pt.y - 64.0)) / h_arena
                        row[f'{name}.Position.X'] = max(0.0, min(1.0, norm_x))
                        row[f'{name}.Position.Y'] = max(0.0, min(1.0, norm_y))
                        row[f'{name}.Confidence'] = pt.score
                    else:
                        row[f'{name}.Position.X'] = float('nan')
                        row[f'{name}.Position.Y'] = float('nan')
                        row[f'{name}.Confidence'] = 0.0
        data.append(row)

    if data:
        out_csv = os.path.join(output_dir, f"{c_id}_pose.csv")
        pd.DataFrame(data).sort_values('FrameIndex').to_csv(out_csv, index=False, na_rep='NaN')
        print(f"    Saved: {os.path.basename(out_csv)}")
    else:
        print(f"    No data found for {c_id}.")

print("\nPipeline completed!")
PYEOF